In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("Model loaded successfully!")

Model loaded successfully!


In [2]:
import spacy

def expand_verb(token):
    """
    Expand verb to include auxiliary, negation, and particles
    Example: 'did not work'
    """
    
    indices = [token.i]

    for child in token.children:
        if child.dep_ in ("aux", "neg", "prt"):
            indices.append(child.i)

    start = min(indices)
    end = max(indices) + 1

    return token.doc[start:end].text


def extract_mentions(sentence):

    doc = nlp(sentence)
    mentions = set()

    # Step 1: Extract noun chunks
    for chunk in doc.noun_chunks:
        if not chunk.root.is_stop:
            mentions.add(chunk.text)

    # Step 2: Extract verbs
    for token in doc:
        if token.pos_ == "VERB" and not token.is_stop:
            verb_phrase = expand_verb(token)
            mentions.add(verb_phrase)

    return list(mentions)


# Example
sentence = "The laptop has great battery life but the screen cracked and the warranty did not cover it."

mentions = extract_mentions(sentence)


In [3]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

def cost_matrix_creation(mentions, intents, model):
    # -----------------------------
    # Mentions extracted from text
    # -----------------------------
    # mentions = [
    #     "great quality",
    #     "warranty issue",
    #     "blue color"
    # ]
    
    # -----------------------------
    # Intent set (including No Intent)
    # -----------------------------
    # intents = [
    #     "Praise",
    #     "Complaint",
    #     "Inquiry",
    #     "battery"
    # ]
    
    # -----------------------------
    # Load embedding model
    # -----------------------------
    
    
    # -----------------------------
    # Compute embeddings
    # -----------------------------
    mention_embeddings = model.encode(mentions)
    intent_embeddings = model.encode(intents)
    
    # -----------------------------
    # Compute cosine similarity
    # -----------------------------
    similarity_matrix = cosine_similarity(mention_embeddings, intent_embeddings)
    
    # -----------------------------
    # Convert similarity to cost
    # C = 1 - cosine similarity
    # -----------------------------
    cost_matrix = 1 - similarity_matrix
    
    # -----------------------------
    # Print cost matrix
    # -----------------------------
    print("Cost Matrix (Mention vs Intent):\n")
    
    for i, m in enumerate(mentions):
        row = []
        for j in range(len(intents)):
            row.append(round(cost_matrix[i][j], 3))
        # print(m, ":", row)
    
    # print("\nMatrix Shape:", cost_matrix.shape)
    return cost_matrix

C:\Users\IIM SBP\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pulp
import numpy as np
import pandas as pd


def mention_intent_pair_finding(mentions, intents, lambda_penalty):
    """
    Mention–Intent Assignment Model

    Objective:
        min  Σ_i Σ_j C_ij x_ij + Σ_i λ y_i

    Subject to:
        Σ_j x_ij + y_i = 1       for every mention i
        Σ_i x_ij <= u_j          for every intent j
        x_ij ∈ {0,1}
        y_i  ∈ {0,1}

    Parameters
    ----------
    mentions : list
        Extracted mentions.
    intents : list
        Predefined intent labels.
    lambda_penalty : float
        Penalty for leaving a mention unassigned.

    Returns
    -------
    total_cost : float
        Optimal objective value.
    df_mention_intent : pandas.DataFrame
        Mention–intent assignments.
    """

    # ---------------------------------------------------------
    # Cost matrix
    # C_ij = 1 - cosine_similarity(M_i, I_j)
    # ---------------------------------------------------------
    cost = cost_matrix

    m = len(mentions)   # number of mentions
    n = len(intents)    # number of intents

    penalty = lambda_penalty

    # ---------------------------------------------------------
    # Intent capacity u_j
    #
    # As stated in the paper, an intent can be associated
    # with any number of mentions. Therefore, we set the
    # capacity of every intent equal to the number of mentions.
    #
    # u_j = m
    # ---------------------------------------------------------
    u = {j: m for j in range(n)}

    # ---------------------------------------------------------
    # Optimization model
    # ---------------------------------------------------------
    model = pulp.LpProblem(
        "Mention_Intent_Assignment",
        pulp.LpMinimize
    )

    # ---------------------------------------------------------
    # Decision variables
    #
    # x_ij = 1 if mention i is assigned to intent j
    # y_i  = 1 if mention i remains unassigned
    # ---------------------------------------------------------
    x = pulp.LpVariable.dicts(
        "x",
        ((i, j) for i in range(m) for j in range(n)),
        cat="Binary"
    )

    y = pulp.LpVariable.dicts(
        "y",
        (i for i in range(m)),
        cat="Binary"
    )

    # ---------------------------------------------------------
    # Objective Function — Equation (2)
    #
    # min Σ_i Σ_j C_ij x_ij + Σ_i λ y_i
    # ---------------------------------------------------------
    model += (
        pulp.lpSum(
            cost[i][j] * x[(i, j)]
            for i in range(m)
            for j in range(n)
        )
        +
        pulp.lpSum(
            penalty * y[i]
            for i in range(m)
        )
    )

    # ---------------------------------------------------------
    # Mention Assignment Constraint — Equation (3)
    #
    # Each mention must either:
    #   1. be assigned to exactly one intent, OR
    #   2. remain unassigned.
    #
    # Σ_j x_ij + y_i = 1
    # ---------------------------------------------------------
    for i in range(m):
        model += (
            pulp.lpSum(
                x[(i, j)]
                for j in range(n)
            )
            + y[i]
            == 1
        ), f"Mention_Assignment_{i}"

    # ---------------------------------------------------------
    # Intent Capacity Constraint — Equation (4)
    #
    # Σ_i x_ij <= u_j
    #
    # Multiple mentions can be assigned to the same intent.
    # ---------------------------------------------------------
    for j in range(n):
        model += (
            pulp.lpSum(
                x[(i, j)]
                for i in range(m)
            )
            <= u[j]
        ), f"Intent_Capacity_{j}"

    # ---------------------------------------------------------
    # Binary Constraints — Equation (5)
    #
    # x_ij ∈ {0,1}
    # y_i  ∈ {0,1}
    #
    # Already enforced through cat="Binary".
    # ---------------------------------------------------------

    # ---------------------------------------------------------
    # Solve
    # ---------------------------------------------------------
    model.solve()

    print("Status:", pulp.LpStatus[model.status])
    print()

    # ---------------------------------------------------------
    # Extract solution
    # ---------------------------------------------------------
    assignments = []

    for i in range(m):

        assigned = False

        for j in range(n):

            if pulp.value(x[(i, j)]) == 1:

                print(
                    mentions[i],
                    "-->",
                    intents[j]
                )

                assignments.append({
                    "Mention": mentions[i],
                    "Intent": intents[j]
                })

                assigned = True
                break

        # Mention remains unassigned
        if pulp.value(y[i]) == 1:

            print(
                mentions[i],
                "--> No Intent"
            )

            assignments.append({
                "Mention": mentions[i],
                "Intent": "No Intent"
            })

    # ---------------------------------------------------------
    # Create output DataFrame
    # ---------------------------------------------------------
    df_mention_intent = pd.DataFrame(assignments)

    total_cost = pulp.value(model.objective)

    print("\nTotal Cost:", total_cost)

    return total_cost, df_mention_intent

In [6]:
import pandas as pd

# load dataset ## Only Review with Column Name "text"
df = pd.read_csv("....Path\\MixATIS.csv")

# load categories ## Only category with Column Name "category"
df_Cat = pd.read_csv("....Path\\MixATIS_Categories.csv")
# Step 1: Split categories by '#'
df_Cat['category_split'] = df_Cat['category'].astype(str).apply(lambda x: x.split('#'))

# Step 2: Flatten list of lists
all_categories = [cat.strip() for sublist in df_Cat['category_split'] for cat in sublist]

# Step 3: Get unique categories
unique_categories = sorted(set(all_categories))

# print result
print("Unique Categories:\n", unique_categories)
print("\nTotal Unique Categories:", len(unique_categories))

intents = unique_categories

In [8]:
from datetime import datetime
start_time = datetime.now()

final_df = pd.DataFrame()
model = SentenceTransformer("all-MiniLM-L6-v2")
for sentence in df['text']:
    print("\nsentence : ", sentence)
    mentions = extract_mentions(sentence)
    print("mentions : ", mentions)
    if len(mentions) > 0:
        cost_matrix = cost_matrix_creation(mentions, intents, model)
        lambda_penalty = np.median(np.min(cost_matrix[:, :-1], axis=1))
        obj_val, df_mention_intent = mention_intent_pair_finding(mentions, intents, lambda_penalty)
        Filtered_intent_df = df_mention_intent[df_mention_intent['Intent'] != "No Intent"]
        print ("Filtered_intent_df : \n", Filtered_intent_df)
        final_intents = Filtered_intent_df['Intent'].unique()
        temp = pd.DataFrame({'Sentence':[sentence], 'Intents':[final_intents]})
        final_df = pd.concat([final_df, temp], axis=0)
    else:
        temp = pd.DataFrame({'Sentence':[sentence], 'Intents':['']})
        final_df = pd.concat([final_df, temp], axis=0)
    # break

end_time = datetime.now()

print("Start Time:", start_time)
print("End Time:", end_time)
print("Total Time:", end_time - start_time)



In [9]:
from copy import deepcopy
final_df_ = deepcopy(final_df)
final_df_['Intents'] = final_df_['Intents'].apply(lambda x: ', '.join(x) if isinstance(x, (list, np.ndarray)) else x)
final_df_

In [10]:
final_df_.to_csv("...Path\\MixATIS_Results_on_test_data.csv", index=False)
